# Discrete Prior Evaluation

Audition the full **codec-prior** stack for an experiment. **Just set `CONFIG_NAME`** —
`eval_utils` resolves the DDSP synth, the VQ `LatentCompressor` and the
`PriorDiscrete` checkpoints from `configs/<CONFIG_NAME>.yaml`.

Sections:
1. Print the training stats (prior `val_acc`/`val_loss`, compressor perplexity).
2. **DDSP alone** — autoencode a real chunk (upper bound: no quantization, no prior).
3. **+ VQ compressor** — quantize the controls through the compressor and resynthesise
   (isolates what the discrete codes cost).
4. **Prior generation** — sample new tokens cold-started from the learned START token
   (what the nn~ export does), decode and synthesise.

In [ ]:
CONFIG_NAME  = "harmsworth_stereo"
DEVICE       = "cuda"

PRIOR_CKPT   = "best_acc"   # "best_acc" | "best_loss" | "last"
GEN_SECONDS  = 30.0          # length to generate from the prior
TEMPERATURE  = 1.0          # multinomial sampling temperature
SAMPLING     = "multinomial"  # "multinomial" | "argmax"
SEED         = 0
CHUNK_S      = 8.0          # real-chunk length for the autoencode / compressor comparison

In [ ]:
import os, sys, math
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Audio, display

import eval_utils as E

sns.set_theme(style="darkgrid", context="notebook", palette="muted")
torch.set_grad_enabled(False)
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")

## 1 - Load everything from the config name

In [ ]:
cfg        = E.load_config(CONFIG_NAME)
ddsp       = E.load_ddsp(cfg, device=DEVICE)
compressor = E.load_compressor(cfg, device=DEVICE)
prior      = E.load_prior(cfg, prefer=PRIOR_CKPT, device=DEVICE)

fs         = int(ddsp.fs)
n_channels = int(ddsp.n_channels)
tps        = E.tokens_per_second(ddsp, compressor)
print(f"fs={fs}  n_channels={n_channels}  control_rate={fs/ddsp.resampling_factor:.1f} Hz")
print(f"compressor: {compressor.compression_ratio}x  ->  {tps:.2f} tokens/s")
print(f"prior: num_codebooks={prior.num_codebooks}  codebook_size={prior.codebook_size}  "
      f"start_token_id={prior.start_token_id}  max_len={prior._max_len}")

## 2 - Training stats

`val_acc` is next-token accuracy. Random baseline is `1 / codebook_size`. A large
gap between `acc` (train) and `val_acc` usually means the small blocked val split is
easier/more repetitive than the train set, so read `val_acc` as optimistic.

In [ ]:
def show(stats, keys):
    for k in keys:
        if k in stats:
            s = stats[k]
            print(f"  {k:18s} last={s['last']:.5f}  (min={s['min']:.5f}  max={s['max']:.5f}  n={s['n']})")

try:
    ps = E.prior_stats(cfg)
    print("PRIOR  (random next-token acc = 1/%d = %.4f)" % (prior.codebook_size, 1.0 / prior.codebook_size))
    show(ps, ["val_acc", "acc", "val_loss", "loss", "lr", "epoch"])
    # CE loss is in nats, so perplexity = exp(loss); 1 = perfect, codebook_size = uniform.
    if "val_loss" in ps:
        print(f"  -> val perplexity   = exp(val_loss) = {math.exp(ps['val_loss']['last']):.3f}  "
              f"(1=perfect, {prior.codebook_size}=uniform)")
    if "loss" in ps:
        print(f"  -> train perplexity = exp(loss)     = {math.exp(ps['loss']['last']):.3f}")
except Exception as e:
    print("prior stats unavailable:", e)

try:
    cs = E.compressor_stats(cfg)
    print("\nCOMPRESSOR  (val_loss IS the validation reconstruction MSE; perplexity near codebook_size => good usage)")
    show(cs, ["val_loss", "train_recon_loss", "val_vq_loss", "train_vq_loss",
              "val_perplexity", "train_perplexity"])
    # val_loss is recon-only; train_loss adds vq_weight*vq_loss, so compare like-for-like:
    if "val_loss" in cs and "train_recon_loss" in cs:
        gap = cs["val_loss"]["last"] - cs["train_recon_loss"]["last"]
        print(f"  -> val_recon - train_recon = {gap:+.5f}  (gen. gap; near 0 => not overfit, just capacity-limited)")
except Exception as e:
    print("compressor stats unavailable:", e)

## 3 - A real chunk: DDSP alone vs. + VQ compressor

In [ ]:
dataset = E.load_dataset(cfg, ddsp, chunk_duration_s=CHUNK_S, device=DEVICE)
idx = int(np.random.default_rng(SEED).integers(len(dataset)))
audio, features = dataset[idx]
print(f"chunk {idx} / {len(dataset)}   audio={tuple(audio.shape)}  features={tuple(features.shape)}")

# (a) DDSP alone — autoencode (no quantization, no prior). This is the quality ceiling.
y_ddsp = E.autoencode(ddsp, audio, features)

# (b) + VQ compressor — quantize the controls to codes and resynthesise.
controls = E.build_controls(ddsp, audio, features)
y_vq, codes = E.compressor_roundtrip(ddsp, compressor, controls)

T = min(audio.shape[-1], y_ddsp.shape[-1], y_vq.shape[-1])
x_np      = audio[:, :T].cpu().numpy()
y_ddsp_np = y_ddsp[0, :, :T].cpu().numpy()
y_vq_np   = y_vq[0, :, :T].cpu().numpy()
print(f"compressor codes: {tuple(codes.shape)}  unique={int(codes.unique().numel())} "
      f"/ {prior.codebook_size * prior.num_codebooks}")

In [ ]:
def spectrum(ax, sig, sr, label, color):
    mono = np.mean(sig, axis=0)
    n_fft = 4096
    f = np.fft.rfftfreq(n_fft, d=1.0 / sr)
    mag = 20 * np.log10(np.abs(np.fft.rfft(mono, n=n_fft)) + 1e-8)
    ax.plot(f, mag, color=color, alpha=0.8, lw=0.8, label=label)

fig, ax = plt.subplots(figsize=(12, 4))
spectrum(ax, x_np,      fs, "Original",        sns.color_palette()[0])
spectrum(ax, y_ddsp_np, fs, "DDSP autoencode", sns.color_palette()[2])
spectrum(ax, y_vq_np,   fs, "DDSP + VQ",       sns.color_palette()[3])
ax.set(xlabel="Frequency (Hz)", ylabel="Magnitude (dB)", xlim=(0, fs / 2),
       title=f"Chunk {idx}: reconstruction spectra")
ax.legend(frameon=True); sns.despine(fig=fig); plt.tight_layout(); plt.show()

print("Original:");            display(Audio(x_np, rate=fs))
print("DDSP autoencode:");     display(Audio(y_ddsp_np, rate=fs))
print("DDSP + VQ compressor:"); display(Audio(y_vq_np, rate=fs))

## 4 - Control space: original vs. VQ-autoencoded

The prior models the **VQ codes**, so its ceiling is whatever the compressor's
quantization preserves. Here we overlay each control dimension (DDSP features and
latents) before vs. after the VQ round-trip — flattened or badly-tracked
trajectories here cap what any prior can do downstream.

In [ ]:
controls_vq = compressor.decode_codes(codes, output_len=controls.shape[1])
controls_vq = controls_vq[:, :, : (ddsp.feature_dim + ddsp.latent_size)]
c_orig = controls[0].cpu().numpy()      # [T, feature_dim+latent]
c_vq   = controls_vq[0].cpu().numpy()

fd, ls = ddsp.feature_dim, ddsp.latent_size
names = [f"feature[{i}]" for i in range(fd)] + [f"latent[{i}]" for i in range(ls)]
t = np.arange(c_orig.shape[0]) / (fs / ddsp.resampling_factor)

n_rows = fd + ls
fig, ax = plt.subplots(n_rows, 1, figsize=(14, 2.2 * n_rows), sharex=True)
if n_rows == 1:
    ax = [ax]
for d in range(n_rows):
    ax[d].plot(t, c_orig[:, d], color=sns.color_palette()[0], lw=0.9, label="original")
    ax[d].plot(t, c_vq[:, d], color=sns.color_palette()[3], lw=0.9, alpha=0.8, label="VQ autoencoded")
    ax[d].set_ylabel(names[d]); ax[d].grid(True, alpha=0.3)
ax[0].legend(ncol=2, fontsize=8); ax[-1].set_xlabel("time (s)")
mae = np.abs(c_orig - c_vq).mean(axis=0)
recon_mse = float(np.mean((c_orig - c_vq) ** 2))
fig.suptitle(f"Chunk {idx}: control space, original vs. VQ-autoencoded  "
             f"(recon MSE={recon_mse:.5f})", y=1.0)
sns.despine(fig=fig); plt.tight_layout(); plt.show()

# This live recon MSE is exactly the quantity the compressor reports as val_loss.
print(f"VQ control reconstruction MSE (this chunk): {recon_mse:.5f}")
print(f"per-dim MAE [{', '.join(names)}]: {np.array2string(mae, precision=4)}")

## 5 - Prior generation (cold-started from the START token)

Mirrors the nn~ export: position 0 is the learned START token, then the prior
autoregressively samples tokens, which the compressor decodes back to control
sequences for the DDSP synth. Re-run for a different random take.

In [ ]:
n_tokens = max(1, int(math.ceil(GEN_SECONDS * tps)))
tokens = E.sample_tokens(prior, n_tokens, temperature=TEMPERATURE, sampling=SAMPLING, device=DEVICE)
uniq = int(tokens.unique().numel())
print(f"generated {n_tokens} tokens (~{GEN_SECONDS:.1f}s)  unique={uniq} "
      f"/ {prior.codebook_size * prior.num_codebooks}  (low => collapsed/loopy)")

controls_gen = compressor.decode_codes(tokens)[:, :, : (ddsp.feature_dim + ddsp.latent_size)]
y_gen = E.synth_from_controls(ddsp, controls_gen)
mx = y_gen.abs().max().clamp(min=1e-8)
y_gen_np = (y_gen / mx).clamp(-1, 1)[0].cpu().numpy()   # [n_ch, T]
print("generated audio:", y_gen_np.shape)

In [ ]:
ctrl = controls_gen[0].cpu().numpy()       # [T, feature_dim+latent]
fd, ls = ddsp.feature_dim, ddsp.latent_size
t = np.arange(ctrl.shape[0]) / (fs / ddsp.resampling_factor)

fig, ax = plt.subplots(3, 1, figsize=(14, 9))
for i in range(fd):
    ax[0].plot(t, ctrl[:, i], lw=0.8, label=f"feature[{i}]")
ax[0].set(title="Generated DDSP features (control rate)"); ax[0].legend(fontsize=8)
for i in range(ls):
    ax[1].plot(t, ctrl[:, fd + i], lw=0.8, label=f"latent[{i}]")
ax[1].set(title="Generated DDSP latents (control rate)"); ax[1].legend(fontsize=8)
ax[2].hist(tokens.cpu().numpy().reshape(-1), bins=prior.codebook_size)
ax[2].set(title="Generated token histogram (all codebooks)", xlabel="token id", ylabel="count")
for a in ax[:2]: a.set_xlabel("time (s)"); a.grid(True, alpha=0.3)
sns.despine(fig=fig); plt.tight_layout(); plt.show()

print("Prior generation:"); display(Audio(y_gen_np, rate=fs))